In [0]:
BASE_URL = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"
BASE_PATH = "/Volumes/workspace/default/olist_data/"

FILES = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

for file_name in FILES:
    dbutils.fs.cp(BASE_URL + file_name, BASE_PATH + file_name)
    print(file_name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/olist_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist_data/olist_customers_dataset.csv,olist_customers_dataset.csv,8562270,1790419619000
dbfs:/Volumes/workspace/default/olist_data/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1790419621000
dbfs:/Volumes/workspace/default/olist_data/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15007623,1790419622000
dbfs:/Volumes/workspace/default/olist_data/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5647783,1790419624000
dbfs:/Volumes/workspace/default/olist_data/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1790419625000
dbfs:/Volumes/workspace/default/olist_data/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1790419627000
dbfs:/Volumes/workspace/default/olist_data/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1790419628000
dbfs:/Volumes/workspace/default/olist_data/olist_sellers_dataset.csv,olist_sellers_dataset.csv,163589,1790419629000
dbfs:/Volumes/workspace/default/olist_data/product_category_name_translation.csv,product_category_name_translation.csv,2613,1790419629000


In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/workspace/default/olist_data/"

bronze = {}

for file_name in FILES:
    table_name = file_name.replace(".csv", "")
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(BASE_PATH + file_name)
    )
    bronze[table_name] = df
    print(table_name, df.count())

olist_customers_dataset 99441
olist_geolocation_dataset 1000163
olist_order_items_dataset 112650
olist_order_payments_dataset 103886
olist_order_reviews_dataset 104162
olist_orders_dataset 99441
olist_products_dataset 32951
olist_sellers_dataset 3095
product_category_name_translation 71


In [0]:
for table_name, df in bronze.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"debiprasad_olist_bronze_{table_name}")
    )

print("Bronze tables created")

Bronze tables created


In [0]:
spark.sql("SHOW TABLES").show(50, truncate=False)

+--------+---------------------------------------------------------+-----------+
|database|tableName                                                |isTemporary|
+--------+---------------------------------------------------------+-----------+
|default |aarogya_desert_analysis                                  |false      |
|default |aarogya_trust_scored                                     |false      |
|default |debiprasad_olist_bronze_olist_customers_dataset          |false      |
|default |debiprasad_olist_bronze_olist_geolocation_dataset        |false      |
|default |debiprasad_olist_bronze_olist_order_items_dataset        |false      |
|default |debiprasad_olist_bronze_olist_order_payments_dataset     |false      |
|default |debiprasad_olist_bronze_olist_order_reviews_dataset      |false      |
|default |debiprasad_olist_bronze_olist_orders_dataset             |false      |
|default |debiprasad_olist_bronze_olist_products_dataset           |false      |
|default |debiprasad_olist_b

In [0]:
orders = spark.table("debiprasad_olist_bronze_olist_orders_dataset")
items = spark.table("debiprasad_olist_bronze_olist_order_items_dataset")
payments = spark.table("debiprasad_olist_bronze_olist_order_payments_dataset")
reviews = spark.table("debiprasad_olist_bronze_olist_order_reviews_dataset")
customers = spark.table("debiprasad_olist_bronze_olist_customers_dataset")
sellers = spark.table("debiprasad_olist_bronze_olist_sellers_dataset")
products = spark.table("debiprasad_olist_bronze_olist_products_dataset")
translation = spark.table("debiprasad_olist_bronze_product_category_name_translation")

In [0]:
orders_clean = (
    orders
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
)

items_clean = (
    items
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    )
    .dropDuplicates(["order_id", "order_item_id"])
)

payments_clean = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_value"),
        F.sum("payment_installments").alias("payment_installments")
    )
)

reviews_clean = (
    reviews
    .select(
        "order_id",
        F.expr("try_cast(review_score as double)").alias("review_score")
    )
    .groupBy("order_id")
    .agg(F.avg("review_score").alias("review_score"))
)

customers_clean = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
).dropDuplicates(["customer_id"])

sellers_clean = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
).dropDuplicates(["seller_id"])

products_clean = products.select(
    "product_id",
    "product_category_name"
).dropDuplicates(["product_id"])

translation_clean = translation.select(
    "product_category_name",
    "product_category_name_english"
).dropDuplicates(["product_category_name"])

print("Silver preparation complete")

Silver preparation complete


In [0]:
product_data = (
    products_clean.alias("p")
    .join(
        translation_clean.alias("t"),
        F.col("p.product_category_name") == F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("p.product_id").alias("product_id"),
        F.col("p.product_category_name").alias("product_category_name"),
        F.col("t.product_category_name_english").alias("product_category_name_english")
    )
)

silver = (
    orders_clean.alias("o")
    .join(items_clean.alias("i"), "order_id", "left")
    .join(payments_clean.alias("pay"), "order_id", "left")
    .join(reviews_clean.alias("r"), "order_id", "left")
    .join(customers_clean.alias("c"), "customer_id", "left")
    .join(sellers_clean.alias("s"), F.col("i.seller_id") == F.col("s.seller_id"), "left")
    .join(product_data.alias("p"), F.col("i.product_id") == F.col("p.product_id"), "left")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("c.customer_unique_id"),
        F.col("i.order_item_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.col("p.product_category_name"),
        F.col("p.product_category_name_english"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("pay.payment_value"),
        F.col("r.review_score"),
        F.col("o.order_purchase_timestamp"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.order_estimated_delivery_date"),
        F.datediff(
            F.to_date("o.order_delivered_customer_date"),
            F.to_date("o.order_purchase_timestamp")
        ).alias("delivery_days"),
        F.datediff(
            F.to_date("o.order_estimated_delivery_date"),
            F.to_date("o.order_purchase_timestamp")
        ).alias("estimated_delivery_days"),
        F.when(
            F.col("o.order_delivered_customer_date") >
            F.col("o.order_estimated_delivery_date"),
            1
        ).otherwise(0).alias("is_late"),
        F.concat(
            F.year("o.order_purchase_timestamp"),
            F.lit("-Q"),
            F.quarter("o.order_purchase_timestamp")
        ).alias("order_quarter"),
        (
            F.coalesce(F.col("i.price"), F.lit(0.0)) +
            F.coalesce(F.col("i.freight_value"), F.lit(0.0))
        ).alias("revenue")
    )
)

print("Silver dataset created")

Silver dataset created


In [0]:
silver.select(
    "order_id",
    "customer_unique_id",
    "seller_id",
    "product_category_name_english",
    "revenue",
    "delivery_days",
    "is_late",
    "review_score",
    "order_quarter"
).show(10, truncate=False)

+--------------------------------+--------------------------------+--------------------------------+-------------------------------+-------+-------------+-------+------------+-------------+
|order_id                        |customer_unique_id              |seller_id                       |product_category_name_english  |revenue|delivery_days|is_late|review_score|order_quarter|
+--------------------------------+--------------------------------+--------------------------------+-------------------------------+-------+-------------+-------+------------+-------------+
|5ff96c15d0b717ac6ad1f3d77225a350|e2dfa3127fedbbca9707b36304996dab|1900267e848ceeba8fa32d80c1a5f5a8|bed_bath_table                 |32.7   |5            |0      |5.0         |2018-Q3      |
|22d17396ac4c177e0ea83929cd4f845d|ded340acb51cad2b368c3e986d006ba8|ad781527c93d00d89a11eecd9dcad7c1|baby                           |217.78 |16           |0      |5.0         |2017-Q3      |
|0a4a2fccb27bd83a892fa503987a595b|c7a9a76a4b24a7e7

In [0]:
silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_silver_orders")

In [0]:
silver_count = spark.table("debiprasad_olist_silver_orders").count()
print("Silver rows:", silver_count)

Silver rows: 113425


In [0]:
gold_category = (
    silver
    .filter(F.col("product_category_name_english").isNotNull())
    .groupBy("product_category_name_english")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.avg("review_score").alias("avg_review_score"),
        F.avg("is_late").alias("late_rate")
    )
    .orderBy(F.desc("late_rate"))
)

gold_category.show(20, truncate=False)

+---------------------------------+------+------------------+------------------+-------------------+
|product_category_name_english    |orders|avg_delivery_days |avg_review_score  |late_rate          |
+---------------------------------+------+------------------+------------------+-------------------+
|home_comfort_2                   |24    |14.533333333333333|3.6296296296296298|0.16666666666666666|
|furniture_mattress_and_upholstery|38    |14.405405405405405|3.8157894736842106|0.13157894736842105|
|audio                            |350   |13.267955801104973|3.8319444444444444|0.12637362637362637|
|fashion_underwear_beach          |121   |13.606299212598426|3.976923076923077 |0.12213740458015267|
|christmas_supplies               |128   |15.673333333333334|4.02054794520548  |0.11764705882352941|
|books_technical                  |260   |10.577946768060837|4.363636363636363 |0.10861423220973783|
|home_confort                     |397   |13.461538461538462|3.8333333333333335|0.101382488

In [0]:
customer_orders = (
    silver
    .groupBy("customer_unique_id", "order_id", "order_quarter")
    .agg(F.sum("revenue").alias("order_revenue"))
)

customer_counts = (
    customer_orders
    .groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("order_count"))
)

gold_customer = (
    customer_orders
    .join(customer_counts, "customer_unique_id", "left")
    .groupBy("order_quarter")
    .agg(
        F.countDistinct("customer_unique_id").alias("unique_customers"),
        F.sum("order_revenue").alias("total_revenue"),
        F.countDistinct(
            F.when(F.col("order_count") > 1, F.col("customer_unique_id"))
        ).alias("repeat_customers")
    )
    .withColumn(
        "repeat_customer_share",
        F.col("repeat_customers") / F.col("unique_customers")
    )
    .orderBy("order_quarter")
)

gold_customer.show(truncate=False)

+-------------+----------------+------------------+----------------+---------------------+
|order_quarter|unique_customers|total_revenue     |repeat_customers|repeat_customer_share|
+-------------+----------------+------------------+----------------+---------------------+
|2016-Q3      |4               |354.75            |0               |0.0                  |
|2016-Q4      |322             |56828.46000000002 |13              |0.040372670807453416 |
|2017-Q1      |5153            |855517.7000000031 |262             |0.05084416844556569  |
|2017-Q2      |9139            |1501576.2300000107|531             |0.05810263705000547  |
|2017-Q3      |12335           |1973657.8900000292|700             |0.05674908796108634  |
|2017-Q4      |17505           |2814076.7399999825|803             |0.04587260782633533  |
|2018-Q1      |20767           |3250689.8299999647|962             |0.04632349400491164  |
|2018-Q2      |19715           |3333238.7999999477|763             |0.03870149632259701  |

In [0]:
gold_seller = (
    silver
    .groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.avg("is_late").alias("late_rate"),
        F.avg("review_score").alias("avg_review_score"),
        F.sum("revenue").alias("total_revenue")
    )
    .filter(F.col("orders") >= 10)
    .orderBy(F.desc("late_rate"))
)

gold_seller.show(20, truncate=False)

+--------------------------------+------+------------------+-------------------+------------------+------------------+
|seller_id                       |orders|avg_delivery_days |late_rate          |avg_review_score  |total_revenue     |
+--------------------------------+------+------------------+-------------------+------------------+------------------+
|b1b3948701c5c72445495bd161b83a4c|18    |23.357142857142858|0.5                |1.7222222222222223|25185.49          |
|2709af9587499e95e803a6498a5a56e9|26    |12.73913043478261 |0.48936170212765956|2.5652173913043477|1859.0199999999998|
|c37b2059d4f90d4feead554e5246565e|12    |19.875            |0.4375             |3.75              |1061.28           |
|f76a3b1349b6df1ee875d1f3fa4340f0|24    |16.875            |0.375              |3.4583333333333335|4748.98           |
|02d35243ea2e497335cd0f076b45675d|14    |25.75             |0.375              |2.6875            |6399.95           |
|c990d6cf976a5718aaedc539f383ab88|10    |21.2222

In [0]:
gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_category_delivery")

gold_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_customer_revenue")

gold_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_seller_reliability")

print("Gold tables saved")

Gold tables saved


In [0]:
for table in [
    "debiprasad_olist_gold_category_delivery",
    "debiprasad_olist_gold_customer_revenue",
    "debiprasad_olist_gold_seller_reliability"
]:
    print(table, spark.table(table).count())

debiprasad_olist_gold_category_delivery 71
debiprasad_olist_gold_customer_revenue 10
debiprasad_olist_gold_seller_reliability 1272


In [0]:
for table in [
    "debiprasad_olist_gold_category_delivery",
    "debiprasad_olist_gold_customer_revenue",
    "debiprasad_olist_gold_seller_reliability"
]:
    print("\n", table)
    spark.table(table).show(5, truncate=False)
    


 debiprasad_olist_gold_category_delivery
+---------------------------------+------+------------------+------------------+-------------------+
|product_category_name_english    |orders|avg_delivery_days |avg_review_score  |late_rate          |
+---------------------------------+------+------------------+------------------+-------------------+
|home_comfort_2                   |24    |14.533333333333333|3.6296296296296298|0.16666666666666666|
|furniture_mattress_and_upholstery|38    |14.405405405405405|3.8157894736842106|0.13157894736842105|
|audio                            |350   |13.267955801104973|3.8319444444444444|0.12637362637362637|
|fashion_underwear_beach          |121   |13.606299212598426|3.976923076923077 |0.12213740458015267|
|christmas_supplies               |128   |15.673333333333334|4.02054794520548  |0.11764705882352941|
+---------------------------------+------+------------------+------------------+-------------------+
only showing top 5 rows

 debiprasad_olist_gold_c

In [0]:
gold_tables = {
    "category_delivery": "debiprasad_olist_gold_category_delivery",
    "customer_revenue": "debiprasad_olist_gold_customer_revenue",
    "seller_reliability": "debiprasad_olist_gold_seller_reliability"
}

EXPORT_BASE = "/Volumes/workspace/default/olist_exports/"

for name, table in gold_tables.items():
    path = EXPORT_BASE + name
    (
        spark.table(table)
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(path)
    )
    print(name, path)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8795729356640761>, line 17
      9 for name, table in gold_tables.items():
     10     path = EXPORT_BASE + name
     11     (
     12         spark.table(table)
     13         .coalesce(1)
     14         .write
     15         .mode("overwrite")
     16         .option("header", True)
---> 17         .csv(path)
     18     )
     19     print(name, path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:922, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    903 self.mode(mode)
    904 self._set_opts(
    905     compression=compression,
    906     sep=sep,
   (...)    920     

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.olist_exports;
SHOW VOLUMES;

database,volume_name
default,olist_data
default,olist_exports


In [0]:
gold_tables = {
    "category_delivery": "debiprasad_olist_gold_category_delivery",
    "customer_revenue": "debiprasad_olist_gold_customer_revenue",
    "seller_reliability": "debiprasad_olist_gold_seller_reliability"
}

EXPORT_BASE = "/Volumes/workspace/default/olist_exports/"

for name, table in gold_tables.items():
    (
        spark.table(table)
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(EXPORT_BASE + name)
    )
    print(name, "exported")

category_delivery exported
customer_revenue exported
seller_reliability exported


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/olist_exports/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist_exports/category_delivery/,category_delivery/,0,1790421013437
dbfs:/Volumes/workspace/default/olist_exports/customer_revenue/,customer_revenue/,0,1790421013437
dbfs:/Volumes/workspace/default/olist_exports/seller_reliability/,seller_reliability/,0,1790421013437


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/olist_exports/category_delivery/"))
display(dbutils.fs.ls("/Volumes/workspace/default/olist_exports/customer_revenue/"))
display(dbutils.fs.ls("/Volumes/workspace/default/olist_exports/seller_reliability/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist_exports/category_delivery/_SUCCESS,_SUCCESS,0,1790421000000
dbfs:/Volumes/workspace/default/olist_exports/category_delivery/_committed_5359509363021450349,_committed_5359509363021450349,113,1790421000000
dbfs:/Volumes/workspace/default/olist_exports/category_delivery/_started_5359509363021450349,_started_5359509363021450349,0,1790420999000
dbfs:/Volumes/workspace/default/olist_exports/category_delivery/part-00000-tid-5359509363021450349-bcd3d59d-d55d-4223-a2c5-ab57c6388fd2-731-1-c000.csv,part-00000-tid-5359509363021450349-bcd3d59d-d55d-4223-a2c5-ab57c6388fd2-731-1-c000.csv,5387,1790421000000


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist_exports/customer_revenue/_SUCCESS,_SUCCESS,0,1790421001000
dbfs:/Volumes/workspace/default/olist_exports/customer_revenue/_committed_4583418251035217506,_committed_4583418251035217506,113,1790421001000
dbfs:/Volumes/workspace/default/olist_exports/customer_revenue/_started_4583418251035217506,_started_4583418251035217506,0,1790421001000
dbfs:/Volumes/workspace/default/olist_exports/customer_revenue/part-00000-tid-4583418251035217506-9d4316ee-e578-4723-b58f-3da2af7d0955-732-1-c000.csv,part-00000-tid-4583418251035217506-9d4316ee-e578-4723-b58f-3da2af7d0955-732-1-c000.csv,569,1790421001000


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist_exports/seller_reliability/_SUCCESS,_SUCCESS,0,1790421003000
dbfs:/Volumes/workspace/default/olist_exports/seller_reliability/_committed_6952733095208067444,_committed_6952733095208067444,113,1790421003000
dbfs:/Volumes/workspace/default/olist_exports/seller_reliability/_started_6952733095208067444,_started_6952733095208067444,0,1790421002000
dbfs:/Volumes/workspace/default/olist_exports/seller_reliability/part-00000-tid-6952733095208067444-cc07efa8-6513-48e6-a81e-4d87e4318d9e-733-1-c000.csv,part-00000-tid-6952733095208067444-cc07efa8-6513-48e6-a81e-4d87e4318d9e-733-1-c000.csv,121234,1790421002000
